# 1. Understand the debt

Unit: America's Debt Crisis. Concept level: **understand → analyze → challenge → capstone**.

Data: the course dataset lives in one deterministic DuckDB file, `../../data/analytics.duckdb`, seeded
from `seed/seed.sql` (identical inside the Docker CLI). Every number here is
stable: rerun any notebook years from now and it reproduces the same answers,
because the seed uses fixed anchors (the course video's figures) plus
deterministic noise -- never the random number generator.

Workflow: run cells top to bottom. A `# TASK` comment marks a cell you should
edit; answer in the markdown cell just below when asked.

## Heads-up reading
The numbers below are the ones the course video leads with:
* **~$40T** total national debt at the start of fiscal 2026 -- *doubled* in ten years, *quadrupled* in twenty.
* **~$3B a day** of net interest on the federal debt.
* Net interest **3.2% of GDP** and climbing.
* A **30-year Treasury auction at 5.3%** in 2026 -- the most expensive run since 2001.
* Per the CBO-style projection, interest sits above nominal growth until the **early 2030s** -- the "breathing room" thesis tested in notebook 2.

Five questions this notebook answers: how big is the debt, how fast is it growing, what does the interest bill do to the budget, who sets the price, and what does "growth fixes it" even mean?

### 1.1 The dataset

| table | rows | what it holds |
|---|---|---|
| `gdp` | 32 | 1995-2026 nominal GDP (billions) |
| `debt` | 32 | total debt + held-by-public as % GDP |
| `interest` | 32 | net interest bill and % of GDP |
| `interest_outlook` | 11 | 2026-2036 projected net interest |
| `deficit` | 50 | 1977-2026 receipts/outlays, deficits, primary-surplus flags |
| `spending` | 36 | 2000-2026 federal outlays by category |
| `thirty_year_yields` | 32 | annualized 30-year auction yields |
| `monthly_30y_yield_2025` | 12 | 2025 monthly yields (the run-up to 5.3%) |
| `energy` | 12 | US production 2017-2028 (crude + natural gas, M boe/d) |
| `growth_quarters` | 14 | 2023Q1-2026Q2 real GDP growth |

In [ ]:
# Bootstrap: imports + connection
%matplotlib inline
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "../../scripts")   # the g-vs-r model used in 02-03
import model as m

DB = "../../data/analytics.duckdb"
con = duckdb.connect(DB)

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.figsize": (8, 4.4),
                     "axes.grid": True, "grid.alpha": 0.35,
                     "font.size": 10})

In [ ]:
tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema='main' ORDER BY table_name"
).fetchall()
print("tables:", ", ".join(t for (t,) in tables))

In [ ]:
debt = con.execute("SELECT year, debt_total_billions FROM debt ORDER BY year").fetchdf()
gdp = con.execute("SELECT year, nominal_gdp_billions FROM gdp ORDER BY year").fetchdf()

fig, ax = plt.subplots()
ax.plot(debt.year, debt.debt_total_billions, label="total debt ($B)")
ax.plot(gdp.year, gdp.nominal_gdp_billions, label="nominal GDP ($B)")
ax.set(xlabel="fiscal year", ylabel="trillions of dollars", title="National debt and GDP")
ax.set_yticklabels([f"${v/1000:.0f}T" for v in ax.get_yticks()])
ax.legend()
plt.show()

for y in (2006, 2016, 2026):
    d = debt.set_index("year").loc[y, "debt_total_billions"]
    prev10 = debt.set_index("year").loc[y - 10, "debt_total_billions"]
    print(f"{y}: ${d:.0f}B  (x{d / prev10:.2f} versus {y-10})")

### 1.2 The interest bill: ~$3 billion a *day*

In [ ]:
intr = con.execute(
    "SELECT year, net_interest_billions, interest_pct_gdp FROM interest ORDER BY year"
).fetchdf()
i26 = intr.set_index("year").loc[2026]
daily = i26.net_interest_billions / 365
print(f"2026: interest {i26.interest_pct_gdp}% of GDP -> ~${daily:.2f}B/day, ${daily*30:.0f}B/month")
assert 2.5 <= daily <= 3.5, "the ~$3B/day anchor"

fig, ax = plt.subplots()
ax.plot(intr.year, intr.interest_pct_gdp, ls="--", marker="o")
ax.axhline(i26.interest_pct_gdp, color="C3", alpha=0.4)
ax.set(xlabel="year", ylabel="% of GDP", title="Net interest share of GDP")
plt.show()

### 1.3 The 30-year: who prices the debt?

In [ ]:
yld = con.execute("SELECT year, avg_30y_pct FROM thirty_year_yields ORDER BY year").fetchdf()
fig, ax = plt.subplots()
ax.plot(yld.year, yld.avg_30y_pct, marker="o", ms=3)
ax.axhline(5.3, color="C3", ls="--", alpha=0.4)
ax.annotate("5.3% sale (2026)", xy=(2026, 5.3), xytext=(2012, 6.2),
            arrowprops=dict(arrowstyle="->"))
ax.set(xlabel="year", ylabel="30-year Treasury yield (%)",
       title="The market price of the fiscal path")
plt.show()

m25 = con.execute("SELECT month, avg_30y_pct FROM monthly_30y_yield_2025 ORDER BY month").fetchdf()
print("2025 peak month:", m25.sort_values("avg_30y_pct").tail(1).values[0])

### 1.4 Where the budget goes

In [ ]:
cp = con.execute(
    "SELECT category, SUM(amount_billions) AS total_billions FROM spending "
    "WHERE fiscal_year = 2026 GROUP BY category ORDER BY total_billions DESC"
).fetchdf()
others = cp.category == "All other"
cp = pd.concat([cp[~others], cp[others]])
cp["pct"] = cp.total_billions / cp.total_billions.sum() * 100
fig, ax = plt.subplots()
bars = ax.barh(cp.category, cp.pct)
ax.invert_yaxis()
for b, p in zip(bars, cp.pct):
    ax.text(b.get_width() + 0.4, b.get_y() + b.get_height()/2, f"{p:.0f}%", va="center")
ax.set(xlabel="% of 2026 outlays", title="Where the 2026 budget goes")
plt.show()
big5 = cp[~others].pct.sum()
print(f"Top-5 named categories = {big5:.0f}% of outlays; interest is in the top few")
assert 74 <= big5 <= 82, "the ~78% top-five share"

### 1.5 The 'growth fixes it' lever -- and where it stalls

In [ ]:
e = con.execute(
    "SELECT year, crude_mbpd, natural_gas_mboed, total_boe_mbpd "
    "FROM energy ORDER BY year"
).fetchdf()
fig, ax = plt.subplots()
ax.plot(e.year, e.total_boe_mbpd, marker="o", ms=3, label="total (M boe/d)")
ax.plot(e.year, e.crude_mbpd, ls="--", label="crude only")
ax.set(xlabel="year", ylabel="million boe/day", title="The energy lever in the 333 plan")
ax.legend()
plt.show()
print("2028 target ~18.7 -> actual in data:",
      round(e.set_index("year").loc[2028, "total_boe_mbpd"], 1))

### 1.6 Checkpoint

1. How many **times** debt grew between **2006-2016** and **2016-2026**`? Look at the printed ratios.
2. The ratio of held-by-public debt to GDP in 2026 is above 100% -- true or false?  (Check `debt`.)
3. Which single category is **not** in the top-5 but still eats ~20% of 2026 outlays?  (Notice interest is _inside_ the top group on the chart.)
4. **Why** does a 5.3% **30-year** yield matter more than a 2-year yield for the debt story?  One sentence below.

> **Answer to 4:** the 30-year locks in the government's borrowing cost for a generation; the market, not the CBO, decides what that cost is.

---
End of notebook 1. Next: `02-forecast-scenarios` -- what the g-vs-r projections say.